# Practica Unidad 3 - Generador de Titulares con Miniature GPT

**Alumno:** Raul Ramirez Adarve

**Asignatura:** Aprendizaje Automatico II

**Objetivo:** Implementar un modelo Transformer desde cero para generar titulares de noticias en espanol, usando tokenizacion a nivel de caracter y la arquitectura GPT (decoder-only).

---

## Parte 1: Preparacion del Entorno

Configuramos el entorno en Google Colab con GPU activada. Necesitamos TensorFlow y Keras para construir el Transformer, y NumPy para la manipulacion de datos numericos.

El dataset contiene 1,079 titulares en espanol de fuentes periodisticas. Cada linea es un titular independiente, lo que nos da un corpus compacto ideal para entrenar un modelo de lenguaje a nivel de caracter.

In [ ]:
# Verificar GPU en Colab
import tensorflow as tf
print("GPU disponible:", tf.config.list_physical_devices('GPU'))

In [ ]:
import numpy as np
import keras
from keras import layers

In [ ]:
# Descargar dataset de titulares
# En Google Colab: !gdown 199dxi24ln2b-_S4mhH2sgpr3nvxmoxZN -O titulares.txt
# En local, descargamos con gdown como libreria
import os

dataset_path = 'titulares.txt'
if not os.path.exists(dataset_path):
    import gdown
    gdown.download(id='199dxi24ln2b-_S4mhH2sgpr3nvxmoxZN', output=dataset_path)
    print("Dataset descargado.")
else:
    print("Dataset ya disponible.")

In [ ]:
with open('titulares.txt', 'r', encoding='utf-8') as f:
    text = f.read()

print(f"Caracteres totales: {len(text)}")
print(f"Titulares (aprox): {text.count(chr(10))}")
print(f"\nMuestra:\n{text[:500]}")

---

## Parte 2: Tokenizacion a Nivel de Caracter

Existen varias estrategias de tokenizacion:

| Estrategia | Vocabulario | Secuencias | Usado por |
|------------|-------------|------------|----------|
| Caracter | Muy pequeno (~70-100) | Largas | Esta practica |
| Subpalabra (BPE) | Medio (~30K-50K) | Medias | GPT, BERT, LLMs modernos |
| Palabra | Muy grande (~100K+) | Cortas | Modelos clasicos |

Usamos **tokenizacion a nivel de caracter** porque simplifica la implementacion y nos permite ver como el modelo aprende desde lo mas basico: primero a formar palabras, luego frases con estructura gramatical.

El proceso consiste en crear un vocabulario con todos los caracteres unicos del texto y mapear cada caracter a un indice numerico (y viceversa).

In [ ]:
# Crear vocabulario ordenado con todos los caracteres unicos
vocab = sorted(set(text))
vocab_size = len(vocab)
print(f"Vocabulario: {vocab_size} caracteres unicos")
print(f"Caracteres: {vocab}")

In [ ]:
# Mapeos caracter <-> indice
char_to_idx = {ch: i for i, ch in enumerate(vocab)}
idx_to_char = {i: ch for i, ch in enumerate(vocab)}


def encode(s):
    """Convierte un string en una lista de indices."""
    return [char_to_idx[c] for c in s]


def decode(ids):
    """Convierte una lista de indices en un string."""
    return ''.join([idx_to_char[i] for i in ids])


# Test de encode/decode con texto del propio dataset
test_str = "nueva ley de energia"
encoded = encode(test_str)
decoded = decode(encoded)
print(f"Original:    '{test_str}'")
print(f"Encoded:     {encoded}")
print(f"Decoded:     '{decoded}'")
print(f"Roundtrip OK: {test_str == decoded}")

---

## Parte 3: Preparar Datos de Entrenamiento

Para entrenar un modelo de lenguaje autoregresivo usamos **teacher forcing**: la entrada es una secuencia de tokens y la salida esperada es la misma secuencia desplazada una posicion.

Ejemplo con `"Hola"` → `[H, o, l, a]`:
- **Entrada (X):** `[H, o, l]`
- **Salida (y):** `[o, l, a]`

En cada posicion, el modelo aprende a predecir el siguiente caracter dado todo el contexto anterior.

Usamos `SEQ_LENGTH = 80` como ventana de contexto (un titular tipico tiene 40-100 caracteres) y `BATCH_SIZE = 64` para entrenamiento eficiente en GPU.

In [ ]:
SEQ_LENGTH = 80
BATCH_SIZE = 64

# Tokenizar todo el texto
tokens = np.array(encode(text))
print(f"Tokens totales: {tokens.shape[0]}")

In [ ]:
def crear_secuencias(tokens, seq_len):
    """Crea pares (X, y) donde y es X desplazado una posicion."""
    X, y = [], []
    for i in range(len(tokens) - seq_len):
        X.append(tokens[i:i + seq_len])
        y.append(tokens[i + 1:i + seq_len + 1])
    return np.array(X), np.array(y)


X, y = crear_secuencias(tokens, SEQ_LENGTH)
print(f"Secuencias de entrenamiento: {X.shape}")
print(f"Shape X: {X.shape} | Shape y: {y.shape}")
print(f"\nEjemplo X[0]: '{decode(X[0])}'")
print(f"Ejemplo y[0]: '{decode(y[0])}'")

In [ ]:
# Crear dataset de TensorFlow con shuffle, batching y prefetch
dataset = tf.data.Dataset.from_tensor_slices((X, y))
dataset = dataset.shuffle(10000).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

print(f"Batches por epoca: {len(dataset)}")

---

## Parte 4: Componentes del Transformer

Implementamos los dos componentes fundamentales de la arquitectura Transformer:

### 4.1 Token & Position Embedding

Los Transformers no tienen recurrencia (como las RNN) ni convoluciones (como las CNN), asi que **no saben el orden de los tokens** por defecto. Los embeddings posicionales resuelven esto: sumamos un vector que representa el significado del token con otro que representa su posicion en la secuencia.

Usamos embeddings posicionales **aprendidos** (como GPT), en contraposicion a los sinusoidales fijos del paper original "Attention is All You Need".

In [ ]:
class TokenAndPositionEmbedding(layers.Layer):
    """Capa de embedding que combina token embedding + position embedding."""

    def __init__(self, maxlen, vocab_size, embed_dim):
        super().__init__()
        self.token_emb = layers.Embedding(
            input_dim=vocab_size, output_dim=embed_dim
        )
        self.pos_emb = layers.Embedding(
            input_dim=maxlen, output_dim=embed_dim
        )

    def call(self, x):
        maxlen = tf.shape(x)[-1]
        positions = tf.range(start=0, limit=maxlen, delta=1)
        positions = self.pos_emb(positions)
        x = self.token_emb(x)
        return x + positions

### 4.2 Bloque Transformer con Atencion Causal

El bloque implementa la secuencia: **Multi-Head Attention → Add & Norm → Feed-Forward → Add & Norm**.

Elementos clave:
- **Multi-Head Attention:** Multiples "cabezas" que atienden a distintas partes de la secuencia en paralelo, capturando relaciones sintacticas y semanticas diferentes.
- **Mascara causal:** Matriz triangular inferior que impide que la posicion `i` vea tokens futuros (`i+1, i+2, ...`). Es lo que hace al modelo autoregresivo.
- **Feed-Forward Network (FFN):** Dos capas densas con activacion GELU. Procesa cada posicion de forma independiente.
- **Layer Norm + Residual:** Conexiones residuales (`inputs + output`) que estabilizan el entrenamiento y permiten apilar bloques sin que los gradientes desaparezcan.

In [ ]:
class TransformerBlock(layers.Layer):
    """Bloque Transformer con atencion causal, FFN y conexiones residuales."""

    def __init__(self, embed_dim, num_heads, ff_dim, dropout=0.1):
        super().__init__()
        self.att = layers.MultiHeadAttention(
            num_heads=num_heads, key_dim=embed_dim
        )
        self.ffn = keras.Sequential([
            layers.Dense(ff_dim, activation="gelu"),
            layers.Dense(embed_dim),
        ])
        self.layernorm1 = layers.LayerNormalization(epsilon=1e-6)
        self.layernorm2 = layers.LayerNormalization(epsilon=1e-6)
        self.dropout1 = layers.Dropout(dropout)
        self.dropout2 = layers.Dropout(dropout)

    def causal_attention_mask(self, batch_size, seq_len):
        """Genera mascara triangular inferior para atencion causal."""
        i = tf.range(seq_len)[:, tf.newaxis]
        j = tf.range(seq_len)
        mask = tf.cast(i >= j, dtype=tf.float32)
        mask = tf.reshape(mask, [1, 1, seq_len, seq_len])
        return tf.tile(mask, [batch_size, 1, 1, 1])

    def call(self, inputs, training=False):
        batch_size = tf.shape(inputs)[0]
        seq_len = tf.shape(inputs)[1]
        mask = self.causal_attention_mask(batch_size, seq_len)

        # Multi-Head Attention + residual + norm
        attn_output = self.att(inputs, inputs, attention_mask=mask)
        attn_output = self.dropout1(attn_output, training=training)
        out1 = self.layernorm1(inputs + attn_output)

        # Feed-Forward + residual + norm
        ffn_output = self.ffn(out1)
        ffn_output = self.dropout2(ffn_output, training=training)
        return self.layernorm2(out1 + ffn_output)

---

## Parte 5: Modelo Completo

Ensamblamos los componentes en un modelo GPT completo:

1. **Input** → secuencia de indices de caracteres
2. **Token + Position Embedding** → vectores densos con informacion de posicion
3. **N bloques Transformer** → procesamiento con atencion causal
4. **Dense + Softmax** → distribucion de probabilidad sobre el vocabulario

| Hiperparametro | Valor | Descripcion |
|----------------|-------|-------------|
| `EMBED_DIM` | 256 | Dimension de los embeddings |
| `NUM_HEADS` | 4 | Cabezas de atencion (cada una de dim 64) |
| `FF_DIM` | 512 | Dimension interna de la FFN |
| `NUM_BLOCKS` | 4 | Bloques Transformer apilados |

In [ ]:
EMBED_DIM = 256
NUM_HEADS = 4
FF_DIM = 512
NUM_BLOCKS = 4


def crear_modelo():
    """Crea el modelo miniature GPT completo."""
    inputs = layers.Input(shape=(SEQ_LENGTH,), dtype=tf.int32)
    x = TokenAndPositionEmbedding(SEQ_LENGTH, vocab_size, EMBED_DIM)(inputs)
    for _ in range(NUM_BLOCKS):
        x = TransformerBlock(EMBED_DIM, NUM_HEADS, FF_DIM)(x)
    outputs = layers.Dense(vocab_size, activation="softmax")(x)
    return keras.Model(inputs=inputs, outputs=outputs)


model = crear_modelo()
model.summary()

---

## Parte 6: Entrenamiento

Entrenamos con **sparse categorical crossentropy** como funcion de perdida, que es la adecuada cuando las etiquetas son indices enteros (no one-hot).

Callbacks utilizados:
- **EarlyStopping:** Detiene el entrenamiento si la perdida no mejora en 3 epocas consecutivas, y restaura los mejores pesos.
- **ReduceLROnPlateau:** Reduce el learning rate a la mitad si la perdida se estanca durante 2 epocas.

In [ ]:
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

callbacks = [
    keras.callbacks.EarlyStopping(patience=3, restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(factor=0.5, patience=2),
]

# En CPU usamos menos epochs; en GPU (Colab) se puede subir a 30
history = model.fit(dataset, epochs=15, callbacks=callbacks)

### Curvas de Entrenamiento

Visualizamos la evolucion de la perdida y la precision a lo largo de las epocas para verificar que el modelo esta aprendiendo correctamente.

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Perdida
axes[0].plot(history.history['loss'], label='Loss', color='steelblue')
axes[0].set_title('Perdida durante el entrenamiento')
axes[0].set_xlabel('Epoca')
axes[0].set_ylabel('Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Precision
axes[1].plot(history.history['accuracy'], label='Accuracy', color='seagreen')
axes[1].set_title('Precision durante el entrenamiento')
axes[1].set_xlabel('Epoca')
axes[1].set_ylabel('Accuracy')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---

## Parte 7: Generacion de Texto

La generacion es **autoregresiva**: el modelo predice un caracter, lo anade a la secuencia, y repite. El parametro **temperatura** controla la aleatoriedad:

- **Temperatura 0.5** → Conservador, elige caracteres muy probables. Texto coherente pero repetitivo.
- **Temperatura 1.0** → Balance entre creatividad y coherencia.
- **Temperatura 1.5** → Mas aleatorio, combinaciones novedosas pero puede ser incoherente.

In [ ]:
def generar_texto(model, inicio, longitud=100, temperatura=1.0):
    """Genera texto de forma autoregresiva a partir de un inicio dado."""
    generado = list(encode(inicio))

    for _ in range(longitud):
        # Tomar los ultimos SEQ_LENGTH tokens como contexto
        input_seq = generado[-SEQ_LENGTH:]
        input_seq = np.array(input_seq)[np.newaxis, :]

        # Padding si la secuencia es mas corta que SEQ_LENGTH
        if len(input_seq[0]) < SEQ_LENGTH:
            pad_len = SEQ_LENGTH - len(input_seq[0])
            input_seq = np.pad(input_seq, ((0, 0), (pad_len, 0)))

        # Predecir distribucion del siguiente caracter
        preds = model.predict(input_seq, verbose=0)[0, -1, :]

        # Aplicar temperatura
        preds = np.log(preds + 1e-10) / temperatura
        preds = np.exp(preds) / np.sum(np.exp(preds))

        # Samplear siguiente caracter
        next_idx = np.random.choice(len(preds), p=preds)
        generado.append(next_idx)

        # Parar en salto de linea (fin de titular)
        if idx_to_char[next_idx] == '\n':
            break

    return decode(generado)

### Generacion con diferentes temperaturas

Probamos el mismo texto de inicio con tres temperaturas para observar el efecto en la generacion.

In [ ]:
print("=" * 60)
print("GENERACION CON DIFERENTES TEMPERATURAS")
print("Texto de inicio: 'el gobierno '")
print("=" * 60)

for temp in [0.5, 1.0, 1.5]:
    print(f"\n--- Temperatura {temp} ---")
    for i in range(3):
        resultado = generar_texto(model, "el gobierno ", temperatura=temp)
        print(f"  {i+1}. {resultado.strip()}")

### Experimentacion con diferentes textos de inicio

In [ ]:
inicios = ["la economia ", "un nuevo ", "el presidente ", "argentina ", "se espera "]

print("=" * 60)
print("GENERACION CON DIFERENTES INICIOS (temperatura=0.8)")
print("=" * 60)

for inicio in inicios:
    print(f"\nInicio: '{inicio}'")
    for i in range(3):
        resultado = generar_texto(model, inicio, temperatura=0.8)
        print(f"  {i+1}. {resultado.strip()}")

---

## Analisis y Reflexion

### Patrones capturados por el modelo

A pesar de ser un modelo pequeno entrenado a nivel de caracter con un corpus reducido (~1,079 titulares), el modelo consigue aprender varios patrones:

1. **Estructura de palabras en espanol:** El modelo aprende a formar palabras reales con acentos, la "n" con tilde, y combinaciones de consonantes y vocales propias del espanol.
2. **Patrones de titulares:** Tiende a generar frases cortas y directas con la estructura tipica de titulares periodisticos (sujeto + verbo + complemento).
3. **Vocabulario tematico:** Reproduce vocabulario frecuente en noticias: "gobierno", "economia", "nuevo", "presidente", etc.
4. **Longitud adecuada:** Los titulares generados tienden a tener una longitud similar a los del dataset de entrenamiento.

### Efecto de la temperatura

| Temperatura | Comportamiento observado |
|-------------|-------------------------|
| 0.5 | Titulares coherentes pero repetitivos. Tiende a generar las mismas frases. Pocas sorpresas. |
| 1.0 | Buen equilibrio. Genera variedad manteniendo coherencia a nivel de palabras. |
| 1.5 | Alta variabilidad. Aparecen palabras inventadas o combinaciones inusuales. Menos legible. |

### Limitaciones

1. **Dataset pequeno:** 1,079 titulares es muy poco. El modelo memoriza patrones del corpus en vez de generalizar. Con mas datos, la calidad mejoraria significativamente.
2. **Tokenizacion por caracter:** El modelo necesita aprender a formar palabras desde cero, lo que es ineficiente. Un tokenizador BPE (como usan GPT o BERT) seria mucho mas eficiente.
3. **Sin validacion:** No hemos separado datos de validacion, asi que no podemos medir overfitting rigurosamente. Las curvas de entrenamiento solo muestran la perdida en train.
4. **Contexto limitado:** Con `SEQ_LENGTH = 80` caracteres, el modelo no puede capturar dependencias que excedan esa ventana.
5. **Sin coherencia semantica garantizada:** El modelo genera texto estadisticamente plausible pero no entiende el significado. Puede generar titulares gramaticalmente correctos pero semanticamente absurdos.

### Conexion con modelos reales

Este mini-GPT comparte la arquitectura fundamental con GPT-4, Claude o Gemini:
- Atencion causal (decoder-only)
- Embeddings posicionales
- Bloques Transformer apilados
- Generacion autoregresiva

La diferencia esta en la escala: donde nuestro modelo tiene ~1M de parametros y se entrena con ~1K titulares, GPT-4 tiene cientos de miles de millones de parametros y se entrena con billones de tokens. Esto demuestra que la arquitectura escala: los mismos principios que funcionan aqui son los que producen los LLMs de ultima generacion.